In [1]:
import json
import tempfile
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta, timezone

import duckdb
import requests

DB_PATH = "/Users/arnarfreyrerlingsson/Desktop/Polymarket_bitcoin_5min/db/polymarket.duckdb"
GAMMA_MARKETS = "https://gamma-api.polymarket.com/markets"
_local = threading.local()  # one requests.Session per worker thread
TMP_FILE = f"{tempfile.gettempdir()}/bitcoin_5m_markets_batch.json"

# table column -> type (rows are flattened to these keys by flatten_market)
COLUMNS = {
    "MARKET_ID": "VARCHAR", "CONDITION_ID": "VARCHAR", "QUESTION_ID": "VARCHAR", "SLUG": "VARCHAR",
    "EVENT_ID": "VARCHAR", "EVENT_SLUG": "VARCHAR", "QUESTION": "VARCHAR", "RESOLUTION_SOURCE": "VARCHAR",
    "EVENT_START_TIME": "TIMESTAMPTZ", "END_DATE": "TIMESTAMPTZ", "START_DATE": "TIMESTAMPTZ",
    "CREATED_AT": "TIMESTAMPTZ", "CLOSED_TIME": "TIMESTAMPTZ", "UMA_END_DATE": "TIMESTAMPTZ",
    "UP_TOKEN_ID": "VARCHAR", "DOWN_TOKEN_ID": "VARCHAR",
    "UP_PRICE": "DOUBLE", "DOWN_PRICE": "DOUBLE", "RESULT": "VARCHAR",
    "UMA_RESOLUTION_STATUS": "VARCHAR", "AUTOMATICALLY_RESOLVED": "BOOLEAN",
    "ACTIVE": "BOOLEAN", "CLOSED": "BOOLEAN",
    "VOLUME": "DOUBLE", "LAST_TRADE_PRICE": "DOUBLE",
    "ORDER_PRICE_MIN_TICK_SIZE": "DOUBLE", "ORDER_MIN_SIZE": "DOUBLE",
    "FEES_ENABLED": "BOOLEAN", "FEE_TYPE": "VARCHAR", "FEE_RATE": "DOUBLE", "FEE_EXPONENT": "DOUBLE",
    "FEE_TAKER_ONLY": "BOOLEAN", "FEE_REBATE_RATE": "DOUBLE",
    "MAKER_BASE_FEE": "INTEGER", "TAKER_BASE_FEE": "INTEGER",
}
# Markets are written to a temp JSON file and bulk-loaded from it, same as the trades pipeline.
INSERT = (
    f"INSERT INTO bitcoin_5m_markets ({', '.join(COLUMNS)}) "
    f"SELECT {', '.join(COLUMNS)} "
    f"FROM read_json('{TMP_FILE}', format = 'array', columns = "
    f"{{{', '.join(f'{c}: {t!r}' for c, t in COLUMNS.items())}}})"
)


def get_session():
    """Return this thread's requests.Session (Session isn't guaranteed thread-safe)."""
    if not hasattr(_local, "session"):
        _local.session = requests.Session()
    return _local.session


def get_json(url, params):
    """GET with retries; backs off on rate limits (429) and errors."""
    for attempt in range(8):
        try:
            resp = get_session().get(url, params=params, timeout=30)
            if resp.status_code == 429:
                raise requests.HTTPError("429 Too Many Requests")
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException:
            if attempt == 7:
                raise
            time.sleep(min(2 ** attempt, 30))


def day_markets(day):
    """Return the Gamma market objects for the 288 five-minute markets of a UTC day."""
    first = int(day.timestamp())
    slugs = [f"btc-updown-5m-{first + k * 300}" for k in range(288)]
    markets = []
    for j in range(0, len(slugs), 50):
        batch = slugs[j:j + 50]
        params = [("closed", "true"), ("limit", len(batch))] + [("slug", s) for s in batch]
        markets += get_json(GAMMA_MARKETS, params)
    return sorted(markets, key=lambda m: m["slug"])


def flatten_market(m):
    """Gamma market object -> one table row (JSON-string fields split into columns)."""
    outcomes = json.loads(m.get("outcomes") or "[]")
    prices = [float(p) for p in json.loads(m.get("outcomePrices") or "[]")]
    tokens = json.loads(m.get("clobTokenIds") or "[]")
    up, down = outcomes.index("Up"), outcomes.index("Down")
    event = (m.get("events") or [{}])[0]
    fees = m.get("feeSchedule") or {}
    return {
        "MARKET_ID": m.get("id"), "CONDITION_ID": m["conditionId"], "QUESTION_ID": m.get("questionID"),
        "SLUG": m["slug"], "EVENT_ID": event.get("id"), "EVENT_SLUG": event.get("slug"),
        "QUESTION": m.get("question"), "RESOLUTION_SOURCE": m.get("resolutionSource"),
        "EVENT_START_TIME": m.get("eventStartTime"), "END_DATE": m.get("endDate"),
        "START_DATE": m.get("startDate"), "CREATED_AT": m.get("createdAt"),
        "CLOSED_TIME": m.get("closedTime"), "UMA_END_DATE": m.get("umaEndDate"),
        "UP_TOKEN_ID": tokens[up] if tokens else None, "DOWN_TOKEN_ID": tokens[down] if tokens else None,
        "UP_PRICE": prices[up] if prices else None, "DOWN_PRICE": prices[down] if prices else None,
        "RESULT": outcomes[prices.index(1.0)] if 1.0 in prices else None,
        "UMA_RESOLUTION_STATUS": m.get("umaResolutionStatus"),
        "AUTOMATICALLY_RESOLVED": m.get("automaticallyResolved"),
        "ACTIVE": m.get("active"), "CLOSED": m.get("closed"),
        "VOLUME": m.get("volumeNum"), "LAST_TRADE_PRICE": m.get("lastTradePrice"),
        "ORDER_PRICE_MIN_TICK_SIZE": m.get("orderPriceMinTickSize"), "ORDER_MIN_SIZE": m.get("orderMinSize"),
        "FEES_ENABLED": m.get("feesEnabled"), "FEE_TYPE": m.get("feeType"),
        "FEE_RATE": fees.get("rate"), "FEE_EXPONENT": fees.get("exponent"),
        "FEE_TAKER_ONLY": fees.get("takerOnly"), "FEE_REBATE_RATE": fees.get("rebateRate"),
        "MAKER_BASE_FEE": m.get("makerBaseFee"), "TAKER_BASE_FEE": m.get("takerBaseFee"),
    }


def insert_markets(con, markets):
    """Bulk-insert markets."""
    if markets:
        with open(TMP_FILE, "w") as f:
            json.dump([flatten_market(m) for m in markets], f)
        con.execute(INSERT)


def fmt(seconds):
    return str(timedelta(seconds=int(seconds)))


In [2]:
DATE_START = '2026-06-12'
DATE_END = '2026-08-01'
n_runners = 8     # parallel API requests


In [3]:
# Days are UTC. Each day has 288 markets; they are fetched and stored one day at a time.
start = datetime.fromisoformat(DATE_START).replace(tzinfo=timezone.utc)
end = datetime.fromisoformat(DATE_END).replace(tzinfo=timezone.utc)
days = [start + timedelta(days=i) for i in range((end - start).days + 1)]

t0 = time.time()
con = duckdb.connect(DB_PATH)

# Delete the whole date range once up front, so re-running never duplicates rows.
# Slugs compare correctly as text since the unix timestamps all have the same length.
first_slug = f"btc-updown-5m-{int(start.timestamp())}"
after_last_slug = f"btc-updown-5m-{int((end + timedelta(days=1)).timestamp())}"
deleted = con.execute("DELETE FROM bitcoin_5m_markets WHERE SLUG >= ? AND SLUG < ?",
                      [first_slug, after_last_slug]).fetchone()[0]
print(f"deleted {deleted} existing rows for {DATE_START} to {DATE_END}")

# Days are fetched n_runners at a time in worker threads; pool.map yields them in order,
# and all DuckDB writes stay on this thread.
with ThreadPoolExecutor(n_runners) as pool:
    for i, (day, markets) in enumerate(zip(days, pool.map(day_markets, days)), 1):
        insert_markets(con, markets)

        elapsed = time.time() - t0
        eta = elapsed / i * (len(days) - i)
        print(f"[day {i}/{len(days)}] {day:%Y-%m-%d} | markets: {len(markets)}/288 | "
              f"elapsed {fmt(elapsed)} | ETA {fmt(eta)}")

total = con.execute("SELECT COUNT(*) FROM bitcoin_5m_markets").fetchone()[0]
print(f"=== done | table total: {total} rows")

con.close()


deleted 0 existing rows for 2026-06-12 to 2026-08-01
[day 1/51] 2026-06-12 | markets: 288/288 | elapsed 0:00:01 | ETA 0:00:55
[day 2/51] 2026-06-13 | markets: 288/288 | elapsed 0:00:01 | ETA 0:00:44
[day 3/51] 2026-06-14 | markets: 288/288 | elapsed 0:00:02 | ETA 0:00:41
[day 4/51] 2026-06-15 | markets: 288/288 | elapsed 0:00:03 | ETA 0:00:39
[day 5/51] 2026-06-16 | markets: 288/288 | elapsed 0:00:04 | ETA 0:00:37
[day 6/51] 2026-06-17 | markets: 283/288 | elapsed 0:00:04 | ETA 0:00:36
[day 7/51] 2026-06-18 | markets: 288/288 | elapsed 0:00:05 | ETA 0:00:35
[day 8/51] 2026-06-19 | markets: 288/288 | elapsed 0:00:06 | ETA 0:00:34
[day 9/51] 2026-06-20 | markets: 287/288 | elapsed 0:00:07 | ETA 0:00:33
[day 10/51] 2026-06-21 | markets: 288/288 | elapsed 0:00:07 | ETA 0:00:32
[day 11/51] 2026-06-22 | markets: 288/288 | elapsed 0:00:08 | ETA 0:00:31
[day 12/51] 2026-06-23 | markets: 288/288 | elapsed 0:00:09 | ETA 0:00:30
[day 13/51] 2026-06-24 | markets: 288/288 | elapsed 0:00:10 | ETA 0:

In [4]:
con.close()